In [7]:
 !pip install langchain langchain_community pypdf pymupdf sentence_transformers chromadb


Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain-0.3.30-py3-none-any.whl.metadata (6.4 kB)
  Using cached pypdf-6.18.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached pymupdf-1.26.5-cp39-abi3-macosx_11_0_arm64.whl.metadata (3.4 kB)
  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-macosx_10_12_universal2.whl.metadata (10 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
 

In [35]:
from langchain_core.documents import Document

In [36]:
sample=Document(
    page_content="hello",
    metadata={"source":"a.com"}
)

In [37]:
sample

Document(metadata={'source': 'a.com'}, page_content='hello')

In [38]:
# FOR TEXT DATA

from langchain_community.document_loaders.text import TextLoader
loader=TextLoader("data/python.txt",encoding="utf-8")

In [39]:
document=loader.load()
document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [40]:
# # FOR PDF DATA

from langchain_community.document_loaders.pdf import PyPDFLoader
# pdf_loader=PyPDFLoader("data/research2.pdf")
# pdf_document=pdf_loader.load()
# pdf_document

# Ingestion Pipeline

In [41]:
import os 
from langchain_community.document_loaders.pdf import PyPDFLoader

In [42]:
def load_all_pdfs():
    folder_path="data/pdfs"
    num_docs=0
    all_docs=[]
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path=os.path.join(folder_path,filename)
            loader=PyPDFLoader(pdf_path)
            doc=loader.load()
            
            all_docs.extend(doc)
            num_docs+=1
            
    print("Total pdfs:",num_docs)
    print("Total pages:",len(all_docs))
    return all_docs
    

In [43]:
all_pdf_documents=load_all_pdfs()

Total pdfs: 1
Total pages: 23


In [44]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

# chunking

In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [46]:
def split_docs(documents,chunk_size=500,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunked_docs=text_splitter.split_documents(documents)
    return chunked_docs

In [47]:
chunks=split_docs(all_pdf_documents)

In [48]:
len(chunks)

82

In [49]:
import os

print(os.listdir("data/pdfs"))

['research.pdf', '.ipynb_checkpoints']


# Embeddings

In [50]:
from sentence_transformers import SentenceTransformer

In [51]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=SentenceTransformer(self.model_name)
        print("LOADING MODEL WITH ",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("embedding shape",embeddings.shape)
        return embeddings
    

In [52]:
emb_manager=EmbeddingManager()

LOADING MODEL WITH  384


# Vector store

In [53]:
import chromadb
import uuid

In [54]:
class VectorStoreManager:
    def __init__(self,persist_directory="data/vector_store"
                ,collection_name="pdf_documents"):
        self.persist_directory=persist_directory
        self.collection_name=collection_name
        self.collection=None
        self.client=None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok=True)
        #create a client
        self.client=chromadb.PersistentClient(path=self.persist_directory)
        # create collection
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"Vector store collection for embeddings in Rag"},
        )
        print("Initialized Vector Store with collection name:",self.collection_name)
        print("Docs in collection:",self.collection.count())
        
    def add_documents(self,documents,embeddings):
        if len(documents)!=len(embeddings):
            raise ValueError("No. of documents does not match no. of embeddings")

        # we will store->ids ,embeddings,documents,metadata
        ids=[]
        all_metadata=[]
        documents_content=[]
        embeddings_list=[]


        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_len"]=len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )
        print("TOTAL DOCUMENTS added  IN vector store",len(documents_content))
        print("Docs in collection",self.collection.count())

In [55]:
vector_store=VectorStoreManager()

Initialized Vector Store with collection name: pdf_documents
Docs in collection: 82


<!-- steps
data->documents->chunks->embeddings->store in vector store -->

In [56]:
# steps
# data->documents->chunks->embeddings->store in vector store

In [57]:
text =[doc.page_content for doc in chunks]
embeddings=emb_manager.generate_embeddings(text)
vector_store.add_documents(chunks,embeddings)


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 11.09it/s]


embedding shape (82, 384)
TOTAL DOCUMENTS added  IN vector store 82
Docs in collection 164


# RETRIEVAL PIPELINE

In [58]:
 from sklearn.metrics.pairwise import cosine_similarity

In [59]:
class RAGRetriever:
    def __init__(self,emb_manager,vector_store):
        self.emb_manager=emb_manager
        self.vector_store=vector_store

    def retrieve(self,query,top_k=5,score_threshold=0.0):
        # 1 query->embedding
        query_embeddings=self.emb_manager.generate_embeddings([query])[0]
        
        # semantic search
        results=self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        retrieved_docs=[]
        if results["documents"] and results["documents"][0]:
            ids=results["ids"][0]
            metadatas=results["metadatas"][0]
            documents=results["documents"][0]
            distances=results["distances"][0]

            for i,(doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
        #cosine similarity
                similarity_score=1-distance
                # print(similarity_score)

                if similarity_score>=score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "document":document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank":i+1
                    })
            print("retrieved docs: ",len(retrieved_docs))        
            
            
        else:
            print("no documents found")
            
        return retrieved_docs
            

In [60]:
rag_retriever=RAGRetriever(emb_manager,vector_store)

In [65]:
rag_retriever.retrieve("WHAT IS RAG")

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 23.20it/s]

embedding shape (1, 384)
retrieved docs:  0


[]

In [63]:
chatgpt=<REDACTED>

NameError: name 'sk' is not defined

In [64]:
groq=<REDACTED>

NameError: name '<REDACTED>' is not defined

# INTEGRATE WITH LLM

### OPEN AI GPT

In [92]:
API_KEY_OPENAI="<REDACTED>"

In [97]:
# !{sys.executable} -m pip install langchain-openai

In [98]:
from langchain_openai import ChatOpenAI

In [141]:
llm=ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.6-luna",
    temperature=0.1,
    max_tokens=1024
)

##### GENERATE OUR RETRIEVAL AUGMENTED OUTPUT

In [142]:
def generate_output(query,rag_retriever,llm,top_k=3):
    results=rag_retriever.retrieve(query,top_k)

    context="\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("NO RELEVANT CONTEXT FOUND FOR GIVEN QUERY")

    prompt=f""" use given context to generate the answer for the query
                Context:{context}
                Query:{query}"""
    response=llm.invoke(prompt) # expects a string(openai)
    return response.content
    

In [158]:
# answer=generate_output("what is rag",rag_retriever,llm)
# print(answer)

# GROQ

In [112]:
# !{sys.executable} -m pip install langchain-groq

In [111]:
from langchain_groq import ChatGroq

In [116]:
API_KEY_GROQ="<REDACTED>"

In [137]:
llm=ChatGroq(
    groq_api_key=API_KEY_GROQ,
    model="qwen/qwen3.8-27b",
    temperature=0.1,
    max_tokens=1024
)

In [138]:
def generate_output(query,rag_retriever,llm,top_k=3):
    results=rag_retriever.retrieve(query,top_k)

    context="\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("NO RELEVANT CONTEXT FOUND FOR GIVEN QUERY")

    prompt=f""" use given context to generate the answer for the query
                Context:{context}
                Query:{query}"""
    response=llm.invoke(prompt.format(context=context,query=query)) # expects a list(groq)
    return response.content
    

In [139]:
answer=generate_output("what is rag",rag_retriever,llm)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved docs:  3
Based on the provided context, **RAG** (Retrieval-Augmented Generation) is a burgeoning field that has experienced swift growth. While the text does not provide a formal dictionary definition, it describes RAG in the following ways:

*   It is a process that involves **paradigms** such as "naive RAG" and "effective RAG framework."
*   It is closely integrated with **Large Language Models (LLMs)**.
*   It is a technical paradigm and research method that has evolved over time, with current efforts focused on mapping its process, charting its evolution, and anticipating future paths.
*   It is evaluated using specific methods, covering 26 tasks, nearly 50 datasets, and various benchmarks and tools.

In essence, the context positions RAG as a rapidly evolving technical approach within the field of LLMs, characterized by different paradigms and a need for systematic evaluation and future innovation.


In [ ]:
print(answer)

In [140]:
answer=generate_output("what is CHUNKING",rag_retriever,llm)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved docs:  0
NO RELEVANT CONTEXT FOUND FOR GIVEN QUERY
Based on the provided context, there is no information available to answer the query "what is CHUNKING." The context section is empty.


# OLLAMA

In [130]:
# !{sys.executable} -m pip install langchain-ollama

In [131]:
from langchain_ollama import ChatOllama

In [150]:
llm=ChatOllama(

    model="llama3.2",
    temperature=0.1,
    max_tokens=1024
)

In [151]:
def generate_output(query,rag_retriever,llm,top_k=3):
    results=rag_retriever.retrieve(query,top_k)

    context="\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("NO RELEVANT CONTEXT FOUND FOR GIVEN QUERY")

    prompt=f""" use given context to generate the answer for the query
                Context:{context}
                Query:{query}"""
    response=llm.invoke(prompt) # expects a string
    return response.content
    

In [152]:
answer=generate_output("what is RAG",rag_retriever,llm)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved docs:  3
Based on the given context, RAG stands for "Reinforcement Alignment of Generative Models". It is a field of research that aims to align the behavior of generative models (such as language models) with human values and goals. The context suggests that RAG is a growing field that has not been systematically synthesized, and the paper aims to fill this gap by mapping out the RAG process, charting its evolution, and anticipating future paths, with a focus on integrating RAG within Large Language Models (LLMs).


# claude

In [147]:
# !{sys.executable} -m pip install langchain-anthropic

In [149]:
from langchain_anthropic import ChatAnthropic

In [153]:
import os

os.environ["ANTHROPIC_API_KEY"] = "your-api-key"

In [154]:
llm=ChatAnthropic(

    model="claude-sonnet-4-5-20250929",
    
    temperature=0.1,
    max_tokens=1024
)

In [155]:
def generate_output(query,rag_retriever,llm,top_k=3):
    results=rag_retriever.retrieve(query,top_k)

    context="\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("NO RELEVANT CONTEXT FOUND FOR GIVEN QUERY")

    prompt=f""" use given context to generate the answer for the query
                Context:{context}
                Query:{query}"""
    response=llm.invoke(prompt) # expects a string
    return response.content
    

In [157]:
# answer=generate_output("what is RAG",rag_retriever,llm)
# print(answer)